# Advanced Analytics

This notebook performs four analytical tasks:

1. Historical and parametric Value at Risk (VaR)
2. Investor cohort analysis
3. Cohort transaction-value analysis
4. Rule-based mutual fund analytical comparison

The analysis uses the cleaned project datasets. The fund comparison is an
analytical scoring mechanism based on measurable characteristics and is not
personalised financial advice.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Notebook may be executed from either the project root or notebooks folder.
ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

TRANSACTIONS = ROOT / "data" / "processed" / "08_investor_transactions_clean.csv"
PERFORMANCE = ROOT / "data" / "processed" / "07_scheme_performance_clean.csv"
SCORECARD = ROOT / "performance" / "fund_scorecard.csv"
OUTPUT = ROOT / "performance"
OUTPUT.mkdir(parents=True, exist_ok=True)

transactions = pd.read_csv(TRANSACTIONS)
performance = pd.read_csv(PERFORMANCE)
scorecard = pd.read_csv(SCORECARD)

transactions["date"] = pd.to_datetime(transactions["date"], errors="coerce")
transactions["amount"] = pd.to_numeric(transactions["amount"], errors="coerce")

numeric_cols = [
    "return_1y", "return_3y", "return_5y", "benchmark_3yr_pct",
    "alpha", "beta", "sharpe_ratio", "sortino_ratio",
    "std_dev_ann_pct", "max_drawdown_pct", "expense_ratio",
    "morningstar_rating"
]

for col in numeric_cols:
    if col in performance.columns:
        performance[col] = pd.to_numeric(performance[col], errors="coerce")

print("Transactions:", transactions.shape)
print("Performance:", performance.shape)
print("Scorecard:", scorecard.shape)


## 1. Value at Risk

VaR estimates a loss threshold over a specified horizon and confidence level.

- **Historical VaR:** empirical percentile of observed returns.
- **Parametric VaR:** uses mean and standard deviation under a normal-distribution assumption.

The calculation below uses the available 1-year fund return observations.


In [ ]:
returns = pd.to_numeric(performance["return_1y"], errors="coerce").dropna() / 100

confidence = 0.95

historical_var = -returns.quantile(1 - confidence)

mu = returns.mean()
sigma = returns.std()
z = -1.6448536269514722

parametric_var = -(mu + z * sigma)

var_summary = pd.DataFrame({
    "method": ["Historical VaR", "Parametric VaR"],
    "confidence": [confidence, confidence],
    "var_loss_pct": [
        historical_var * 100,
        parametric_var * 100
    ]
})

var_summary.to_csv(OUTPUT / "advanced_var_summary.csv", index=False)

print(var_summary)


In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(returns * 100, bins=40, kde=True)

plt.axvline(
    -historical_var * 100,
    color="red",
    linestyle="--",
    label=f"Historical VaR: {historical_var * 100:.2f}%"
)

plt.axvline(
    -parametric_var * 100,
    color="orange",
    linestyle="--",
    label=f"Parametric VaR: {parametric_var * 100:.2f}%"
)

plt.title("1-Year Fund Return Distribution and VaR Thresholds")
plt.xlabel("1-Year Return (%)")
plt.ylabel("Number of Funds")
plt.legend()

plt.tight_layout()
plt.savefig(OUTPUT / "advanced_var_distribution.png", dpi=150)
plt.show()


## 2. Investor Cohort Analysis

Investors are assigned to cohorts based on the month of their first recorded transaction.


In [ ]:
transactions = transactions.dropna(subset=["date"]).copy()

first_transaction = (
    transactions
    .groupby("investor_id")["date"]
    .min()
    .rename("first_transaction_date")
)

transactions = transactions.merge(
    first_transaction,
    on="investor_id",
    how="left"
)

transactions["cohort_month"] = (
    transactions["first_transaction_date"]
    .dt.to_period("M")
    .astype(str)
)

cohort_summary = (
    transactions
    .groupby("cohort_month")
    .agg(
        investors=("investor_id", "nunique"),
        transactions=("transaction_id", "count"),
        transaction_value=("amount", "sum")
    )
    .reset_index()
    .sort_values("cohort_month")
)

cohort_summary.to_csv(
    OUTPUT / "investor_cohort_analysis.csv",
    index=False
)

print(cohort_summary.head(20))


In [ ]:
plot_data = cohort_summary.tail(24)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=plot_data,
    x="cohort_month",
    y="investors",
    color="steelblue"
)

plt.title("Investor Cohort Size by First Transaction Month")
plt.xlabel("Cohort Month")
plt.ylabel("Number of Investors")
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig(OUTPUT / "investor_cohort_size.png", dpi=150)
plt.show()


## 3. Cohort Transaction Value

The following visualisation compares total transaction value across investor cohorts.


In [ ]:
plot_data = cohort_summary.tail(24)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=plot_data,
    x="cohort_month",
    y="transaction_value",
    color="seagreen"
)

plt.title("Transaction Value by Investor Cohort")
plt.xlabel("Cohort Month")
plt.ylabel("Total Transaction Value")
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig(
    OUTPUT / "investor_cohort_transaction_value.png",
    dpi=150
)
plt.show()


## 4. Rule-Based Fund Analytical Comparison

The following score combines measurable fund characteristics:

- 3-year return
- Sharpe ratio
- Alpha
- Expense ratio
- Maximum drawdown

Each component is percentile-ranked across the available funds.
Higher return, Sharpe and alpha contribute positively, while lower expense
ratio and lower drawdown contribute positively.

This is an analytical comparison and does not guarantee future returns.


In [ ]:
rec = performance.copy()

required = [
    "amfi_code",
    "scheme_name",
    "return_3y",
    "sharpe_ratio",
    "alpha",
    "expense_ratio",
    "max_drawdown_pct"
]

rec = rec[[c for c in required if c in rec.columns]].copy()

def percentile_score(series, higher_is_better=True):
    result = series.rank(pct=True, ascending=not higher_is_better)
    return result.fillna(result.median())

rec["return_3y_score"] = percentile_score(rec["return_3y"], True)
rec["sharpe_score"] = percentile_score(rec["sharpe_ratio"], True)
rec["alpha_score"] = percentile_score(rec["alpha"], True)
rec["expense_score"] = percentile_score(rec["expense_ratio"], False)
rec["drawdown_score"] = percentile_score(rec["max_drawdown_pct"], False)

rec["recommendation_score"] = (
    0.25 * rec["return_3y_score"] +
    0.25 * rec["sharpe_score"] +
    0.20 * rec["alpha_score"] +
    0.15 * rec["expense_score"] +
    0.15 * rec["drawdown_score"]
)

recommendations = rec.sort_values(
    "recommendation_score",
    ascending=False
).reset_index(drop=True)

recommendations["analytical_rank"] = (
    recommendations.index + 1
)

recommendations.to_csv(
    OUTPUT / "advanced_fund_recommender.csv",
    index=False
)

print(recommendations.head(10))


In [ ]:
top = recommendations.head(10).sort_values(
    "recommendation_score"
)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=top,
    x="recommendation_score",
    y="scheme_name",
    color="mediumpurple"
)

plt.title("Top 10 Analytical Fund Scores")
plt.xlabel("Recommendation Score")
plt.ylabel("Fund")

plt.tight_layout()
plt.savefig(
    OUTPUT / "advanced_recommender_scores.png",
    dpi=150
)
plt.show()


## Outputs

The notebook produces:

- `advanced_var_summary.csv`
- `advanced_var_distribution.png`
- `investor_cohort_analysis.csv`
- `investor_cohort_size.png`
- `investor_cohort_transaction_value.png`
- `advanced_fund_recommender.csv`
- `advanced_recommender_scores.png`

### Interpretation

VaR provides a downside-risk threshold under stated assumptions. Historical
VaR depends on the observed return distribution, while parametric VaR also
assumes a normal distribution.

Cohort analysis identifies differences in investor acquisition periods and
transaction activity.

The analytical fund comparison converts measurable fund characteristics into
a transparent composite score. It is intended for analytical comparison and
does not guarantee future returns.
